# Battery ride-through: a reproducible first run

This short teaching notebook uses the bundled `generator_failure` synthetic scenario to answer one bounded question: how long can stored battery energy serve a 1,000 kW IT request when utility and generator supply are unavailable? The calculation is an electrical continuity example, not a UPS sizing result or a facility claim.

The notebook runs the repository engine, records its version and actual input digest, and checks the key numbers independently with exact rational arithmetic.

## Goal

By the end, you will have reproduced the default 100 kWh case and two 50 kWh cases:

- Default `generator_failure`: the outage starts at 300 s; 100 kWh × 0.90 discharge efficiency × 0.95 distribution efficiency gives 85.5 kWh at the IT boundary, or 307.8 s of ride-through. Absolute depletion is 607.8 s.
- Default charging enabled with 50 kWh initially: the battery charges to 57.916666... kWh by 300 s, rides through 178.2675 s, and depletes at 478.2675 s.
- Charging disabled with 50 kWh initially: there is no pre-outage charge, so depletion is 453.9 s.

All quantities below retain their units.

## Setup

The computational dependency is the standard library plus `datacenter_twin` from this repository. Jupyter, `nbclient`, and `nbconvert` are execution tools; they are not part of the simulation model. The import search below lets the notebook run from the repository root or from its `docs/examples` directory without printing a machine-specific path.

In [1]:
from dataclasses import replace
from decimal import Decimal, localcontext
from fractions import Fraction
import hashlib
import json
from pathlib import Path
import sys

_repo_root = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                   if (candidate / 'datacenter_twin').is_dir()), None)
if _repo_root is None:
    raise RuntimeError('Run this notebook inside a datacenter-twin-lab checkout')
sys.path.insert(0, str(_repo_root))

from datacenter_twin import __version__
from datacenter_twin.continuity import simulate_continuity
from datacenter_twin.demo import demo_scenario

def fraction_text(value):
    if value.denominator == 1:
        return str(value.numerator)
    with localcontext() as context:
        context.prec = 80
        rendered = format(Decimal(value.numerator) / Decimal(value.denominator), 'f')
    return rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered

def depleted_at(result):
    return next(event['at_s'] for event in result['events']
                if event['action'] == 'battery_depleted')

## Steps

### 1. Run the bundled scenario and record provenance

`generator_failure` makes both utility and generator unavailable at 300 s and restores both at 900 s. The run is deterministic: the engine embeds its version, a SHA-256 digest of the canonical scenario input, and a synthetic quality label.

In [2]:
base_scenario = demo_scenario('generator_failure')
base_run = simulate_continuity(base_scenario).to_dict()
canonical_input = json.dumps(base_scenario.to_dict(), sort_keys=True, separators=(',', ':'))
expected_input_sha256 = hashlib.sha256(canonical_input.encode('utf-8')).hexdigest()

assert base_run['engine_version'] == __version__
assert base_run['input_sha256'] == expected_input_sha256
assert base_run['quality'] == 'synthetic_uncalibrated'

provenance = {
    'engine_version': base_run['engine_version'],
    'input_sha256': base_run['input_sha256'],
    'model': base_run['model'],
    'quality': base_run['quality'],
}
provenance

{'engine_version': '0.3.0a0',
 'input_sha256': '202d1950bc8762c95216904dd9bbc890c1e1cdb51fe3f824e6e3d98f76410195',
 'model': 'single_load_electrical_continuity_v1',
 'quality': 'synthetic_uncalibrated'}

The fixture has a fictional grid tariff and no generator energy rate. The run may report a grid energy charge when grid energy is used, but that value is illustrative and is not a complete cost model. Unknown generator cost, capital cost, non-energy charges, and market calibration remain explicit.

In [3]:
costs = {
    'generator_cost_per_kwh': base_scenario.generator_cost_per_kwh,
    'total_incremental_energy_charge': base_run['summary']['total_incremental_energy_charge'],
    'cost_status': base_run['summary']['cost_status'],
}
assert costs['generator_cost_per_kwh'] is None
costs

{'generator_cost_per_kwh': None,
 'total_incremental_energy_charge': '37.59',
 'cost_status': 'illustrative'}

### 2. Derive the default ride-through independently

The default inputs are 100 kWh stored energy, 0.90 battery discharge efficiency, 0.95 distribution efficiency, and a 1,000 kW IT request. The product of the efficiencies converts stored energy to energy delivered at the IT boundary:

```text
100 kWh × 0.90 × 0.95 = 85.5 kWh
85.5 kWh / 1,000 kW × 3,600 s/h = 307.8 s
300 s outage start + 307.8 s = 607.8 s absolute depletion
```

`Fraction` keeps this check independent of binary floating-point rounding.

In [4]:
stored_energy_kwh = Fraction(100)
discharge_efficiency = Fraction(9, 10)
distribution_efficiency = Fraction(95, 100)
it_demand_kw = Fraction(1000)
outage_start_s = Fraction(300)

delivered_energy_kwh = stored_energy_kwh * discharge_efficiency * distribution_efficiency
ride_through_s = delivered_energy_kwh / it_demand_kw * 3600
depletion_s = outage_start_s + ride_through_s

assert delivered_energy_kwh == Fraction(171, 2)
assert ride_through_s == Fraction(1539, 5)
assert depletion_s == Fraction(3039, 5)

{
    'delivered_energy_kwh': fraction_text(delivered_energy_kwh),
    'ride_through_s': fraction_text(ride_through_s),
    'absolute_depletion_s': fraction_text(depletion_s),
}

{'delivered_energy_kwh': '85.5',
 'ride_through_s': '307.8',
 'absolute_depletion_s': '607.8'}

In [5]:
assert depleted_at(base_run) == fraction_text(depletion_s)
assert base_run['summary']['unserved_duration_s'] == '292.2'
expected_unserved_kwh = it_demand_kw * (Fraction(900) - depletion_s) / 3600
with localcontext() as context:
    context.prec = 100
    expected_unserved_text = format(Decimal(expected_unserved_kwh.numerator) / Decimal(expected_unserved_kwh.denominator), 'f')
assert base_run['summary']['unserved_it_kwh'] == expected_unserved_text
assert base_run['summary']['energy_balance_residual_kwh'] == '0'

{
    'engine_depletion_s': depleted_at(base_run),
    'engine_unserved_duration_s': base_run['summary']['unserved_duration_s'],
    'engine_energy_balance_residual_kwh': base_run['summary']['energy_balance_residual_kwh'],
}

{'engine_depletion_s': '607.8',
 'engine_unserved_duration_s': '292.2',
 'engine_energy_balance_residual_kwh': '0'}

### 3. See why the 50 kWh case charges before the outage

Changing only the initial stored energy to 50 kWh leaves a 100 kW charge limit available while utility power is present. During the first 300 s, the opening balance becomes:

```text
50 kWh + 100 kW × 0.95 × (300 s / 3,600 s/h) = 57.916666... kWh
57.916666... kWh × 0.90 × 0.95 / 1,000 kW × 3,600 s/h = 178.2675 s
300 s + 178.2675 s = 478.2675 s absolute depletion
```

In [6]:
charged50_scenario = replace(base_scenario, battery_initial_kwh=Decimal('50'))
charged50_run = simulate_continuity(charged50_scenario).to_dict()
charged50_at_outage = next(row for row in charged50_run['intervals'] if row['start_s'] == '300')
charged50_expected_opening_kwh = Fraction(50) + Fraction(100) * Fraction(95, 100) * Fraction(300, 3600)
charged50_ride_through_s = charged50_expected_opening_kwh * discharge_efficiency * distribution_efficiency / it_demand_kw * 3600
charged50_depletion_s = outage_start_s + charged50_ride_through_s

with localcontext() as context:
    context.prec = 100
    charged50_expected_opening_text = format(Decimal(charged50_expected_opening_kwh.numerator) / Decimal(charged50_expected_opening_kwh.denominator), 'f')
assert charged50_at_outage['battery_start_kwh'] == charged50_expected_opening_text
assert charged50_ride_through_s == Fraction('178.2675')
assert charged50_depletion_s == Fraction('478.2675')
assert depleted_at(charged50_run) == '478.2675'
assert charged50_run['summary']['energy_balance_residual_kwh'] == '0'

{
    'input_sha256': charged50_run['input_sha256'],
    'battery_start_at_300_s_kwh': charged50_at_outage['battery_start_kwh'],
    'derived_ride_through_s': fraction_text(charged50_ride_through_s),
    'engine_depletion_s': depleted_at(charged50_run),
}

{'input_sha256': '7d6ba4e2919be3714c753ef7ba22f6d278cf57dbc8be8bf5efdfaf6375d48775',
 'battery_start_at_300_s_kwh': '57.91666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666666667',
 'derived_ride_through_s': '178.2675',
 'engine_depletion_s': '478.2675'}

### 4. Contrast a 50 kWh case with charging disabled

This is a separate scenario input. Setting `battery_charge_kw` to zero removes the pre-outage charge, so the battery enters the outage with exactly 50 kWh.

In [7]:
uncharged50_scenario = replace(charged50_scenario, battery_charge_kw=Decimal('0'))
uncharged50_run = simulate_continuity(uncharged50_scenario).to_dict()
uncharged50_at_outage = next(row for row in uncharged50_run['intervals'] if row['start_s'] == '300')

assert uncharged50_at_outage['battery_start_kwh'] == '50'
assert depleted_at(uncharged50_run) == '453.9'
assert uncharged50_run['summary']['energy_balance_residual_kwh'] == '0'

{
    'input_sha256': uncharged50_run['input_sha256'],
    'battery_start_at_300_s_kwh': uncharged50_at_outage['battery_start_kwh'],
    'engine_depletion_s': depleted_at(uncharged50_run),
}

{'input_sha256': '2c05bddf69ee7a54acb3b7d03d1a4d8840da9922758954175a52fff2bd097262',
 'battery_start_at_300_s_kwh': '50',
 'engine_depletion_s': '453.9'}

## Checks

The assertions above independently verify the input digest, engine version, default ride-through and depletion, the 50 kWh charging balance, the 50 kWh no-charge contrast, and a zero energy-balance residual. The hashes shown in the outputs are generated from the actual scenarios executed in this notebook; they should change if a scenario input changes.

## Next Steps

Use the [scenario catalog](../scenarios/index.md) to compare this case with utility loss, path maintenance, and shared-domain failure. For a longer narrative, see the [continuity walkthrough](../tutorials/continuity-walkthrough.md).

This notebook remains a deterministic software teaching example. It does not calibrate a facility, select a battery or UPS, model switching transients, cooling, workload queues, protection, fuel dynamics, aging, reliability probabilities, or physical controls.